# 03 - Feature Engineering

Builds interpretable engineered features on top of the 30 ternary features:
* Statistical aggregates: sum, mean, std, n_suspicious, n_legit, n_neutral,
  phishing_ratio, legit_ratio
* Pairwise interaction products of the top-K features by mutual information


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd, numpy as np
from src.utils import get_config, setup_logging
from src.features import FeatureEngineer
setup_logging()
cfg = get_config()
train = pd.read_csv(cfg.raw_data_dir / cfg.config.files.train_data)
X = train[cfg.features]; y = train[cfg.target_name]
print(f"Original: {X.shape}")


## 1. Mutual information for interaction selection

In [ ]:
from sklearn.feature_selection import mutual_info_classif
mi = mutual_info_classif(X, y, random_state=42, discrete_features=True)
mi_series = pd.Series(mi, index=X.columns).sort_values(ascending=False)
print(mi_series.head(10))


## 2. Fit FeatureEngineer

In [ ]:
eng = FeatureEngineer(
    features=list(cfg.features),
    top_k_interactions=8,
    interaction_features=True,
)
eng.fit(X, y, mi_scores=mi_series)
print(f"Selected {len(eng.interaction_pairs_)} interaction pairs")
print(f"Engineered names: {eng.engineered_names_[:8]} ...")


## 3. Transform and inspect

In [ ]:
X_eng = eng.transform(X)
print(f"Engineered shape: {X_eng.shape}")
X_eng.head()


## 4. MI of engineered features

In [ ]:
mi_eng = mutual_info_classif(X_eng, y, random_state=42, discrete_features=True)
mi_eng_series = pd.Series(mi_eng, index=X_eng.columns).sort_values(ascending=False)
print("Top 15 engineered features by MI:")
print(mi_eng_series.head(15))


## 5. Save the engineer

In [ ]:
eng.save(cfg.preprocessing_dir / "feature_engineer.joblib")
print("Saved feature_engineer.joblib")


## Summary

The `FeatureEngineer` module expands the 30 ternary features to **66 total
features** by adding 8 statistical aggregates and 28 pairwise interaction
products. The top engineered features by mutual information are
`risk_score_std`, `risk_score_sum`, `risk_score_mean`, and `phishing_ratio`
- indicating that aggregate risk metrics carry strong signal beyond the
individual features. The fitted engineer is persisted for reuse at
inference time.
